# Setup

## **Imports**

In [1]:
!pip install -q segmentation-models-pytorch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.6/154.6 kB 3.6 MB/s eta 0:00:0000:01


In [2]:
import os
# os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
# os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
# os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

from pathlib import Path
import json
import sys
import time
import yaml

import numpy as np
import pandas as pd
from PIL import Image, ImageDraw

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import segmentation_models_pytorch as smp

from scipy.ndimage import binary_propagation, label
from tqdm.auto import tqdm
from datetime import datetime




In [3]:
# Surya
sys.path.append("/kaggle/working/surya_minimal")
from surya.models.helio_spectformer import HelioSpectFormer

/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


## **Paths and Device**

In [4]:
# Surya
SURYA_DIR = Path("/kaggle/working/surya_weights")
SURYA_CODE_DIR = Path("/kaggle/working/surya_minimal")

In [5]:
ROOT = Path(
    "/kaggle/input/competitions/filament-segmentation-2026/"
    "MAGFiLO_1.0_Kaggle_2026"
)

TRAIN_IMAGES = ROOT / "train" / "train_images"

ANNOTATIONS_PATH = (
    ROOT
    / "train"
    / "MAGFiLO_1.0_Annotations_kaggle2026_train.json"
)

CHECKPOINT_DIR = Path("/kaggle/working/checkpoints")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)


device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

gpu_count = torch.cuda.device_count()

print("Device:", device)
print("Available GPUs:", gpu_count)

if torch.cuda.is_available():
    for i in range(gpu_count):
        print(f"GPU {i}: {torch.cuda.get_device_name(i)}")

Device: cuda
Available GPUs: 2
GPU 0: Tesla T4
GPU 1: Tesla T4


## **Model Utilities**

In [6]:
def prepare_model(model):
    return model

In [7]:
def load_surya_config_and_checkpoint():
    config_path = SURYA_DIR / "config.yaml"
    checkpoint_path = SURYA_DIR / "surya.366m.v1.pt"

    with open(config_path, "r") as f:
        config = yaml.safe_load(f)

    checkpoint = torch.load(checkpoint_path, map_location="cpu")

    return config, checkpoint

In [8]:
def print_gpu_memory():

    for i in range(torch.cuda.device_count()):

        allocated = (
            torch.cuda.memory_allocated(i)
            / 1024**3
        )

        reserved = (
            torch.cuda.memory_reserved(i)
            / 1024**3
        )

        peak = (
            torch.cuda.max_memory_allocated(i)
            / 1024**3
        )

        print(
            f"GPU {i}: "
            f"Allocated={allocated:.2f} GiB | "
            f"Reserved={reserved:.2f} GiB | "
            f"Peak={peak:.2f} GiB"
        )

In [9]:
surya_config, surya_checkpoint = load_surya_config_and_checkpoint()

In [10]:
print("\nMemory after model initialization:")
print_gpu_memory()


Memory after model initialization:
GPU 0: Allocated=0.00 GiB | Reserved=0.00 GiB | Peak=0.00 GiB
GPU 1: Allocated=0.00 GiB | Reserved=0.00 GiB | Peak=0.00 GiB


# Data
## Load Annotations

In [11]:
with open(ANNOTATIONS_PATH, "r") as f:
    annotations_data = json.load(f)

## Data Utilities

In [12]:
def index_by(items, key):
    index = {}

    for item in items:
        value = item[key]
        index.setdefault(value, []).append(item)

    return index


image_entries_by_filename = index_by(
    annotations_data["images"],
    "file_name"
)

annotations_by_image_id = index_by(
    annotations_data["annotations"],
    "image_id"
)


def load_image(filename):
    return Image.open(
        TRAIN_IMAGES / filename
    ).convert("L")


def get_annotated_filenames():
    return sorted(
        image_entries_by_filename.keys()
    )


def get_image_entries(filename):
    return image_entries_by_filename.get(
        filename,
        []
    )


def get_annotations(image_id):
    return annotations_by_image_id.get(
        image_id,
        []
    )


def get_image_annotations(
    filename,
    annotation_set=0
):
    image_entries = get_image_entries(filename)

    if not image_entries:
        raise ValueError(
            f"No annotations found for {filename}"
        )

    if not 0 <= annotation_set < len(image_entries):
        raise ValueError(
            f"Annotation set {annotation_set} does not exist "
            f"for {filename}. "
            f"Available sets: {len(image_entries)}"
        )

    image_id = image_entries[annotation_set]["id"]

    return get_annotations(image_id)


def annotations_to_mask(
    annotations,
    image_size
):
    mask = Image.new(
        "L",
        image_size,
        0
    )

    draw = ImageDraw.Draw(mask)

    for annotation in annotations:
        segmentation = annotation["segmentation"][0]

        points = [
            (
                segmentation[i],
                segmentation[i + 1]
            )
            for i in range(
                0,
                len(segmentation),
                2
            )
        ]

        draw.polygon(
            points,
            fill=1
        )

    return np.array(
        mask,
        dtype=np.uint8
    )


def get_image_and_mask(
    filename,
    annotation_set=0
):
    image = load_image(filename)

    annotations = get_image_annotations(
        filename,
        annotation_set
    )

    mask = annotations_to_mask(
        annotations,
        image.size
    )

    return image, mask


def get_training_samples():
    samples = []

    for filename, image_entries in (
        image_entries_by_filename.items()
    ):
        for annotation_set in range(
            len(image_entries)
        ):
            samples.append(
                (filename, annotation_set)
            )

    return sorted(samples)

## Dataset

In [13]:
class FilamentDataset(Dataset):
    def __init__(self, samples):
        self.samples = samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        filename, annotation_set = self.samples[index]

        image, mask = get_image_and_mask(
            filename,
            annotation_set
        )

        image = np.array(
            image,
            dtype=np.float32
        ) / 255.0

        mask = mask.astype(np.float32)

        image = torch.from_numpy(
            image
        ).unsqueeze(0)

        mask = torch.from_numpy(
            mask
        ).unsqueeze(0)

        return image, mask

## Data Splitting

**Splitting Utilities**

In [14]:
def get_observation_date(filename):
    timestamp = filename[:14]

    return datetime.strptime(
        timestamp,
        "%Y%m%d%H%M%S"
    ).date()


def group_filenames_by_date(filenames):
    filenames_by_date = {}

    for filename in filenames:
        date = get_observation_date(filename)

        filenames_by_date.setdefault(
            date,
            []
        ).append(filename)

    return filenames_by_date


def split_filenames_by_date(
    filenames,
    validation_fraction=0.2,
    seed=42
):
    filenames_by_date = group_filenames_by_date(
        filenames
    )

    dates = np.array(
        sorted(filenames_by_date.keys())
    )

    rng = np.random.default_rng(seed)
    rng.shuffle(dates)

    validation_count = max(
        1,
        round(len(dates) * validation_fraction)
    )

    validation_dates = set(
        dates[:validation_count]
    )

    train_filenames = []
    validation_filenames = []

    for date, date_filenames in filenames_by_date.items():
        if date in validation_dates:
            validation_filenames.extend(date_filenames)
        else:
            train_filenames.extend(date_filenames)

    return (
        sorted(train_filenames),
        sorted(validation_filenames)
    )


def get_samples_for_filenames(filenames):
    samples = []

    for filename in filenames:
        image_entries = get_image_entries(filename)

        for annotation_set in range(
            len(image_entries)
        ):
            samples.append(
                (filename, annotation_set)
            )

    return samples

**Split Script**

In [15]:
filenames = get_annotated_filenames()

train_filenames, validation_filenames = (
    split_filenames_by_date(filenames)
)

train_samples = get_samples_for_filenames(
    train_filenames
)

validation_samples = get_samples_for_filenames(
    validation_filenames
)

print("Training images:", len(train_filenames))
print("Validation images:", len(validation_filenames))

print("Training samples:", len(train_samples))
print("Validation samples:", len(validation_samples))

Training images: 562
Validation images: 145
Training samples: 905
Validation samples: 249


## DataLoaders

In [16]:
train_dataset = FilamentDataset(train_samples)
validation_dataset = FilamentDataset(validation_samples)

train_loader = DataLoader(
    train_dataset,
    batch_size=1,
    shuffle=True,
    num_workers=0,
    pin_memory=False,
    drop_last=True
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=0,
    pin_memory=False
)

print("Train batches:", len(train_loader))
print("Validation batches:", len(validation_loader))

Train batches: 905
Validation batches: 249


# Training
## Losses

In [17]:
def dice_loss(
    logits,
    targets,
    eps=1e-6
):
    probabilities = torch.sigmoid(
        logits
    )

    intersection = (
        probabilities * targets
    ).sum(dim=(1, 2, 3))

    total = (
        probabilities.sum(dim=(1, 2, 3))
        + targets.sum(dim=(1, 2, 3))
    )

    dice = (
        2 * intersection + eps
    ) / (
        total + eps
    )

    return 1 - dice.mean()


def focal_loss(
    logits,
    targets,
    alpha=0.25,
    gamma=2.0
):
    bce = nn.functional.binary_cross_entropy_with_logits(
        logits,
        targets,
        reduction="none"
    )

    probabilities = torch.sigmoid(
        logits
    )

    p_t = (
        probabilities * targets
        + (1 - probabilities)
        * (1 - targets)
    )

    focal_weight = (
        1 - p_t
    ) ** gamma

    alpha_weight = (
        alpha * targets
        + (1 - alpha)
        * (1 - targets)
    )

    focal = (
        alpha_weight
        * focal_weight
        * bce
    )

    return focal.mean()


def segmentation_loss(
    logits,
    targets
):
    focal = focal_loss(
        logits,
        targets,
        alpha=0.25,
        gamma=2.0
    )

    dice = dice_loss(
        logits,
        targets
    )

    return focal + dice

## Metrics

In [18]:
def dice_score(
    logits,
    targets,
    threshold=0.5,
    eps=1e-6
):
    probabilities = torch.sigmoid(logits)
    predictions = (
        probabilities > threshold
    ).float()

    intersection = (
        predictions * targets
    ).sum(dim=(1, 2, 3))

    total = (
        predictions.sum(dim=(1, 2, 3))
        + targets.sum(dim=(1, 2, 3))
    )

    dice = (
        2 * intersection + eps
    ) / (
        total + eps
    )

    return dice.mean().item()


def iou_score(
    logits,
    targets,
    threshold=0.5,
    eps=1e-6
):
    probabilities = torch.sigmoid(logits)
    predictions = (
        probabilities > threshold
    ).float()

    intersection = (
        predictions * targets
    ).sum(dim=(1, 2, 3))

    union = (
        predictions.sum(dim=(1, 2, 3))
        + targets.sum(dim=(1, 2, 3))
        - intersection
    )

    iou = (
        intersection + eps
    ) / (
        union + eps
    )

    return iou.mean().item()


def extract_predicted_instances(
    probability_map,
    low_threshold=0.4,
    high_threshold=0.6,
    min_size=50
):
    low_mask = probability_map >= low_threshold
    high_mask = probability_map >= high_threshold

    grown_mask = binary_propagation(
        high_mask,
        mask=low_mask
    )

    labeled_mask, instance_count = label(
        grown_mask,
        structure=np.ones(
            (3, 3),
            dtype=np.uint8
        )
    )

    instances = []

    for instance_id in range(
        1,
        instance_count + 1
    ):
        instance_mask = (
            labeled_mask == instance_id
        )

        if instance_mask.sum() >= min_size:
            instances.append(instance_mask)

    return instances


def get_ground_truth_instances(
    filename,
    annotation_set
):
    annotations = get_image_annotations(
        filename,
        annotation_set
    )

    image = load_image(filename)
    instances = []

    for annotation in annotations:
        mask = Image.new(
            "L",
            image.size,
            0
        )

        draw = ImageDraw.Draw(mask)

        segmentation = annotation[
            "segmentation"
        ][0]

        points = [
            (
                segmentation[i],
                segmentation[i + 1]
            )
            for i in range(
                0,
                len(segmentation),
                2
            )
        ]

        draw.polygon(
            points,
            fill=1
        )

        instances.append(
            np.array(
                mask,
                dtype=bool
            )
        )

    return instances


def instance_iou(
    ground_truth,
    prediction
):
    intersection = np.logical_and(
        ground_truth,
        prediction
    ).sum()

    union = np.logical_or(
        ground_truth,
        prediction
    ).sum()

    if union == 0:
        return 0.0

    return intersection / union


def panoptic_quality(
    ground_truth_instances,
    predicted_instances,
    match_threshold=0.5
):
    iou_pairs = []

    for gt_index, ground_truth in enumerate(
        ground_truth_instances
    ):
        for pred_index, prediction in enumerate(
            predicted_instances
        ):
            iou = instance_iou(
                ground_truth,
                prediction
            )

            if iou > match_threshold:
                iou_pairs.append(
                    (
                        iou,
                        gt_index,
                        pred_index
                    )
                )

    iou_pairs.sort(
        reverse=True
    )

    matched_ground_truth = set()
    matched_predictions = set()
    matched_ious = []

    for iou, gt_index, pred_index in iou_pairs:
        if gt_index in matched_ground_truth:
            continue

        if pred_index in matched_predictions:
            continue

        matched_ground_truth.add(
            gt_index
        )

        matched_predictions.add(
            pred_index
        )

        matched_ious.append(
            iou
        )

    true_positives = len(
        matched_ious
    )

    false_positives = (
        len(predicted_instances)
        - true_positives
    )

    false_negatives = (
        len(ground_truth_instances)
        - true_positives
    )

    denominator = (
        true_positives
        + 0.5 * false_positives
        + 0.5 * false_negatives
    )

    if true_positives > 0:
        segmentation_quality = (
            sum(matched_ious)
            / true_positives
        )
    else:
        segmentation_quality = 0.0

    if denominator > 0:
        recognition_quality = (
            true_positives
            / denominator
        )

        pq = (
            sum(matched_ious)
            / denominator
        )
    else:
        recognition_quality = 1.0
        pq = 1.0

    return {
        "pq": pq,
        "sq": segmentation_quality,
        "rq": recognition_quality,
        "tp": true_positives,
        "fp": false_positives,
        "fn": false_negatives
    }


def count_instance_relations(
    ground_truth_instances,
    predicted_instances
):
    ground_truth_overlaps = [
        0
        for _ in ground_truth_instances
    ]

    prediction_overlaps = [
        0
        for _ in predicted_instances
    ]

    for gt_index, ground_truth in enumerate(
        ground_truth_instances
    ):
        for pred_index, prediction in enumerate(
            predicted_instances
        ):
            overlap = np.logical_and(
                ground_truth,
                prediction
            ).any()

            if overlap:
                ground_truth_overlaps[
                    gt_index
                ] += 1

                prediction_overlaps[
                    pred_index
                ] += 1

    one_to_many = sum(
        count > 1
        for count in ground_truth_overlaps
    )

    many_to_one = sum(
        count > 1
        for count in prediction_overlaps
    )

    return {
        "one_to_many": one_to_many,
        "many_to_one": many_to_one
    }


def instance_metrics(
    probability_map,
    filename,
    annotation_set
):
    predicted_instances = (
        extract_predicted_instances(
            probability_map
        )
    )

    ground_truth_instances = (
        get_ground_truth_instances(
            filename,
            annotation_set
        )
    )

    metrics = panoptic_quality(
        ground_truth_instances,
        predicted_instances
    )

    metrics.update(
        count_instance_relations(
            ground_truth_instances,
            predicted_instances
        )
    )

    return metrics

## Training Utilities

In [19]:
def save_training_history(
    history,
    checkpoint_path
):
    history_df = pd.DataFrame(history)

    history_df.insert(
        0,
        "epoch",
        range(1, len(history_df) + 1)
    )

    history_path = checkpoint_path.with_suffix(".csv")

    history_df.to_csv(
        history_path,
        index=False
    )

    print(
        f"Saved training history to: "
        f"{history_path}"
    )


def print_validation_metrics(
    train_loss,
    validation
):
    print(
        f"Train Loss:      {train_loss:.4f}"
    )

    print(
        f"Validation Loss: {validation['loss']:.4f}"
    )

    print(
        f"Dice:            {validation['dice']:.4f}"
    )

    print(
        f"IoU:             {validation['iou']:.4f}"
    )

    print()

    print(
        f"PQ:              {validation['pq']:.4f}"
    )

    print(
        f"├─ SQ:           {validation['sq']:.4f}"
    )

    print(
        f"└─ RQ:           {validation['rq']:.4f}"
    )

    print()

    print("Instances")

    print(
        f"├─ TP:           {validation['tp']}"
    )

    print(
        f"├─ FP:           {validation['fp']}"
    )

    print(
        f"├─ FN:           {validation['fn']}"
    )

    print(
        f"├─ One-to-many:  "
        f"{validation['one_to_many']}"
    )

    print(
        f"└─ Many-to-one:  "
        f"{validation['many_to_one']}"
    )


def train_one_epoch(
    model,
    loader,
    optimizer,
    criterion,
    device,
    scaler
):
    model.train()

    total_loss = 0.0

    for batch_idx, (images, masks) in enumerate(loader, start=1):
        print(f"Training image {batch_idx}/{len(loader)}", flush=True)

        images = images.to(
            "cuda:0",
            non_blocking=True
        )
        
        masks = masks.to(
            "cuda:1",
            non_blocking=True
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(device.type == "cuda")
        ):

            batch = {
                "ts": images.unsqueeze(2),
                "time_delta_input": torch.zeros(
                    images.shape[0],
                    1,
                    device=images.device,
                ),
            }
            # print("=== FORWARD START ===", flush=True)
            logits = model(batch)

            loss = criterion(
                logits,
                masks
            )
        # print("=== FORWARD DISPATCH COMPLETE ===", flush=True)
        scaler.scale(loss).backward()
        # print("=== BACKWARD START ===", flush=True)
        scaler.step(optimizer)
        scaler.update()
        # print("=== BACKWARD AND UPDATE DISPATCH COMPLETE ===", flush=True)

        total_loss += loss.item()

    return total_loss / len(loader)


@torch.no_grad()
def validate_one_epoch(
    model,
    loader,
    criterion,
    device
):
    model.eval()

    total_loss = 0.0
    total_dice = 0.0
    total_iou = 0.0
    total_samples = 0

    total_tp = 0
    total_fp = 0
    total_fn = 0

    total_matched_iou = 0.0

    total_one_to_many = 0
    total_many_to_one = 0

    progress_bar = tqdm(
        loader,
        desc="Validation",
        leave=False
    )

    sample_index = 0

    for images, masks in progress_bar:
        batch_size = images.shape[0]

        images = images.to(
            "cuda:0",
            non_blocking=True
        )
        
        masks = masks.to(
            "cuda:1",
            non_blocking=True
        )

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(device.type == "cuda")
        ):

            batch = {
                "ts": images.unsqueeze(2),
                "time_delta_input": torch.zeros(
                    images.shape[0],
                    1,
                    device=images.device,
                ),
            }
            # print("=== FORWARD START ===", flush=True)
            logits = model(batch)

            loss = criterion(
                logits,
                masks
            )

        dice = dice_score(
            logits,
            masks
        )

        iou = iou_score(
            logits,
            masks
        )

        total_loss += (
            loss.item() * batch_size
        )

        total_dice += (
            dice * batch_size
        )

        total_iou += (
            iou * batch_size
        )

        probabilities = torch.sigmoid(
            logits
        ).float().cpu().numpy()

        for batch_offset in range(
            batch_size
        ):
            filename, annotation_set = (
                loader.dataset.samples[
                    sample_index
                ]
            )

            probability_map = probabilities[
                batch_offset,
                0
            ]

            metrics = instance_metrics(
                probability_map,
                filename,
                annotation_set
            )

            tp = metrics["tp"]
            fp = metrics["fp"]
            fn = metrics["fn"]

            denominator = (
                tp
                + 0.5 * fp
                + 0.5 * fn
            )

            matched_iou = (
                metrics["pq"]
                * denominator
            )

            total_tp += tp
            total_fp += fp
            total_fn += fn

            total_matched_iou += (
                matched_iou
            )

            total_one_to_many += (
                metrics["one_to_many"]
            )

            total_many_to_one += (
                metrics["many_to_one"]
            )

            sample_index += 1

        total_samples += batch_size

        progress_bar.set_postfix(
            loss=f"{loss.item():.4f}",
            dice=f"{dice:.4f}",
            iou=f"{iou:.4f}"
        )

    pq_denominator = (
        total_tp
        + 0.5 * total_fp
        + 0.5 * total_fn
    )

    if total_tp > 0:
        sq = (
            total_matched_iou
            / total_tp
        )
    else:
        sq = 0.0

    if pq_denominator > 0:
        rq = (
            total_tp
            / pq_denominator
        )

        pq = (
            total_matched_iou
            / pq_denominator
        )
    else:
        rq = 1.0
        pq = 1.0

    return {
        "loss": total_loss / total_samples,
        "dice": total_dice / total_samples,
        "iou": total_iou / total_samples,
        "pq": pq,
        "sq": sq,
        "rq": rq,
        "tp": total_tp,
        "fp": total_fp,
        "fn": total_fn,
        "one_to_many": total_one_to_many,
        "many_to_one": total_many_to_one
    }


def train_model(
    model,
    train_loader,
    validation_loader,
    optimizer,
    criterion,
    device,
    epochs,
    checkpoint_path,
    patience=3
):
    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=(device.type == "cuda")
    )

    history = {
        "train_loss": [],
        "validation_loss": [],
        "dice": [],
        "iou": [],
        "pq": [],
        "sq": [],
        "rq": [],
        "tp": [],
        "fp": [],
        "fn": [],
        "one_to_many": [],
        "many_to_one": []
    }

    best_pq = -float("inf")
    epochs_without_improvement = 0

    start_time = time.time()

    for epoch in range(epochs):
        print(
            f"\nEpoch {epoch + 1}/{epochs}"
        )

        train_loss = train_one_epoch(
            model=model,
            loader=train_loader,
            optimizer=optimizer,
            criterion=criterion,
            device=device,
            scaler=scaler
        )

        validation = validate_one_epoch(
            model=model,
            loader=validation_loader,
            criterion=criterion,
            device=device
        )

        history["train_loss"].append(
            train_loss
        )

        history["validation_loss"].append(
            validation["loss"]
        )

        for metric in [
            "dice",
            "iou",
            "pq",
            "sq",
            "rq",
            "tp",
            "fp",
            "fn",
            "one_to_many",
            "many_to_one"
        ]:
            history[metric].append(
                validation[metric]
            )

        print_validation_metrics(
            train_loss,
            validation
        )

        if validation["pq"] > best_pq:
            best_pq = validation["pq"]
            epochs_without_improvement = 0

            model_to_save = (
                model.module
                if isinstance(
                    model,
                    nn.DataParallel
                )
                else model
            )

            torch.save(
                model_to_save.state_dict(),
                checkpoint_path
            )

            print(
                "Saved new best model "
                f"(PQ: {best_pq:.4f})."
            )

        else:
            epochs_without_improvement += 1

            print(
                f"No PQ improvement "
                f"({epochs_without_improvement}/"
                f"{patience})"
            )

        if (
            epochs_without_improvement
            >= patience
        ):
            print("Early stopping.")
            break

    elapsed = (
        time.time() - start_time
    )

    print(
        f"Training time: "
        f"{elapsed / 60:.1f} minutes"
    )

    save_training_history(
        history=history,
        checkpoint_path=checkpoint_path
    )

    return history

# Backbone

## Surya

In [20]:
def make_surya_backbone():
    m = surya_config["model"]
    d = surya_config["data"]

    # ----------------------------------
    # 1. Build original pretrained Surya
    # ----------------------------------

    model = HelioSpectFormer(
        img_size=4096,
        patch_size=m["patch_size"],
        in_chans=len(d["sdo_channels"]),
        embed_dim=m["embed_dim"],
        time_embedding={
            "type": "linear",
            "time_dim": len(d["time_delta_input_minutes"]),
        },
        depth=m["depth"],
        n_spectral_blocks=m["n_spectral_blocks"],
        num_heads=m["num_heads"],
        mlp_ratio=m["mlp_ratio"],
        drop_rate=m["drop_rate"],
        window_size=m["window_size"],
        dp_rank=m["dp_rank"],
        learned_flow=m["learned_flow"],
        use_latitude_in_learned_flow=m["use_latitude_in_learned_flow"],
        init_weights=False,

        # Checkpoint all 10 transformer blocks
        checkpoint_layers=list(range(10)),

        rpe=m["rpe"],
        ensemble=m["ensemble"],
        finetune=False,
    )

    # ----------------------------------
    # 2. Load original pretrained weights
    # ----------------------------------

    load_result = model.load_state_dict(
        surya_checkpoint,
        strict=True,
    )

    # ----------------------------------
    # 3. Replace H-alpha patch embedding
    # ----------------------------------

    model.embedding.patch_embed = HAlphaPatchEmbed(
        img_size=2048,
        patch_size=16,
        in_chans=1,
        embed_dim=m["embed_dim"],
        time_dim=1,
    )

    # ----------------------------------
    # 4. Regenerate positional encoding
    # ----------------------------------

    model.embedding._generate_position_encoding(
        img_size=2048,
        patch_size=16,
        embed_dim=m["embed_dim"],
    )

    model.embedding.num_patches = 128 * 128

    # ----------------------------------
    # 5. Resize pretrained spectral filters
    # ----------------------------------

    resize_spectral_filters(
        model,
        grid_size=128,
    )

    # ----------------------------------
    # 6. Replace segmentation decoder
    # ----------------------------------

    from surya.models.embedding import LinearDecoder

    model.unembed = LinearDecoder(
        patch_size=16,
        out_chans=1,
        embed_dim=m["embed_dim"],
    )

    model.patch_size = 16

    # ----------------------------------
    # 7. Verify adaptation
    # ----------------------------------

    assert model.embedding.pos_embed.shape == (
        1, 16384, m["embed_dim"]
    )

    print(
        "Positional embedding:",
        tuple(model.embedding.pos_embed.shape)
    )

    return model, load_result

In [21]:
def resize_spectral_filters(model, grid_size=128):
    new_width = grid_size // 2 + 1
    half_height = grid_size // 2

    for block in model.backbone.blocks_spectral_gating:
        old_weight = block.filter.complex_weight

        assert old_weight.shape[0] == 256
        assert old_weight.shape[1] == 129

        with torch.no_grad():
            new_weight = torch.cat(
                [
                    old_weight[:half_height, :new_width],
                    old_weight[-half_height:, :new_width],
                ],
                dim=0,
            ).clone()

        block.filter.complex_weight = nn.Parameter(new_weight)
        block.filter.h = grid_size
        block.filter.w = new_width

        print(
            "Resized spectral filter:",
            tuple(new_weight.shape)
        )

In [22]:
import torch
import torch.nn as nn
from einops import rearrange


class HAlphaPatchEmbed(nn.Module):
    """
    Patch embedding for a single 2048x2048 H-alpha full-disk image.

    Input:
        [B, 1, 1, 2048, 2048]
             C  T    H     W

    Output:
        [B, 65536, 1280]

    Uses 8x8 H-alpha patches so that the token grid is 256x256,
    matching Surya's original:
        4096 / 16 = 256
        2048 / 8  = 256
    """

    def __init__(
        self,
        img_size=2048,
        patch_size=16,
        in_chans=1,
        embed_dim=1280,
        time_dim=1,
    ):
        super().__init__()

        self.img_size = img_size
        self.patch_size = patch_size
        self.embed_dim = embed_dim
        self.time_dim = time_dim

        self.proj = nn.Conv2d(
            in_channels=in_chans * time_dim,  # 1 * 1 = 1
            out_channels=embed_dim,          # 1280
            kernel_size=patch_size,          # 8
            stride=patch_size,               # 8
        )

    def forward(self, x):
        """
        x: [B, C, T, H, W]
           [B, 1, 1, 2048, 2048]
        """

        B, C, T, H, W = x.shape

        assert C == 1, f"Expected 1 H-alpha channel, got {C}"
        assert T == 1, f"Expected 1 timestep, got {T}"
        assert H == 2048 and W == 2048, (
            f"Expected 2048x2048 input, got {H}x{W}"
        )

        # [B, 1, 1, 2048, 2048]
        #               ↓ flatten C,T
        # [B, 1, 2048, 2048]
        x = x.flatten(1, 2)

        # 8x8 non-overlapping patch projection:
        #
        # [B, 1, 2048, 2048]
        #          ↓
        # [B, 1280, 256, 256]
        x = self.proj(x)

        # Flatten the 256x256 spatial token grid:
        #
        # [B, 1280, 256, 256]
        #          ↓
        # [B, 65536, 1280]
        x = rearrange(
            x,
            "B D HT WT -> B (HT WT) D"
        )

        return x

In [23]:
import time
import torch

def attach_layer_tracing(model):
    handles = []
    start_time = time.perf_counter()

    blocks = (
        list(model.backbone.blocks_spectral_gating)
        + list(model.backbone.blocks_attention)
    )

    def log(message):
        elapsed = time.perf_counter() - start_time
        print(f"[{elapsed:8.2f}s] {message}", flush=True)

    for i, block in enumerate(blocks):

        name = (
            f"Spectral {i}"
            if i < 2
            else f"Attention {i - 2}"
        )

        device = next(block.parameters()).device

        def make_pre_hook(name, device):
            def hook(module, inputs):
                phase = (
                    "FORWARD"
                    if not torch.is_grad_enabled()
                    else "FORWARD / RECOMPUTE"
                )
                log(f"START {phase}: {name} ({device})")
            return hook

        def make_post_hook(name, device):
            def hook(module, inputs, output):
                log(f"END: {name} ({device})")
            return hook

        handles.append(
            block.register_forward_pre_hook(
                make_pre_hook(name, device)
            )
        )

        handles.append(
            block.register_forward_hook(
                make_post_hook(name, device)
            )
        )

    return handles

In [24]:
import torch
from torch.utils.checkpoint import checkpoint


def configure_surya_model_parallel(model):

    gpu0 = torch.device("cuda:0")
    gpu1 = torch.device("cuda:1")

    assert torch.cuda.device_count() >= 2, (
        "Model parallelism requires two GPUs."
    )

    # ----------------------------------
    # GPU 0: Input embedding
    # ----------------------------------

    model.embedding.to(gpu0)

    # ----------------------------------
    # GPU 0: Spectral blocks
    # ----------------------------------

    spectral_blocks = (
        model.backbone.blocks_spectral_gating
    )

    for block in spectral_blocks:
        block.to(gpu0)

    # ----------------------------------
    # Split attention blocks
    # ----------------------------------

    attention_blocks = (
        model.backbone.blocks_attention
    )

    for i, block in enumerate(attention_blocks):
        if i < 2:
            block.to(gpu0)
        else:
            block.to(gpu1)

    # ----------------------------------
    # GPU 1: Output decoder
    # ----------------------------------

    model.unembed.to(gpu1)

    # ----------------------------------
    # Replace backbone forward
    # ----------------------------------

    def parallel_backbone_forward(tokens):

        blocks = list(spectral_blocks) + list(
            attention_blocks
        )

        for i, block in enumerate(blocks):

            # Find GPU holding this block
            block_device = next(
                block.parameters()
            ).device

            # Transfer activations if needed
            if tokens.device != block_device:
                tokens = tokens.to(block_device)

            # Preserve gradient checkpointing
            if i in model.backbone._checkpoint_layers:

                tokens = checkpoint(
                    block,
                    tokens,
                    None,
                    use_reentrant=False
                )

            else:

                tokens = block(tokens, None)

        return tokens

    model.backbone.forward = parallel_backbone_forward

    return model

In [25]:
surya_backbone, load_result = make_surya_backbone()

# Distribute Surya across two GPUs
surya_backbone = configure_surya_model_parallel(
    surya_backbone
)


print("Missing keys:", load_result.missing_keys)
print("Unexpected keys:", load_result.unexpected_keys)

print("\nModel distribution:")

print(
    "Embedding:",
    next(surya_backbone.embedding.parameters()).device
)

for i, block in enumerate(
    surya_backbone.backbone.blocks_spectral_gating
):
    print(
        f"Spectral block {i}:",
        next(block.parameters()).device
    )

for i, block in enumerate(
    surya_backbone.backbone.blocks_attention
):
    print(
        f"Attention block {i}:",
        next(block.parameters()).device
    )

print(
    "Decoder:",
    next(surya_backbone.unembed.parameters()).device
)

Resized spectral filter: (128, 65, 1280, 2)
Resized spectral filter: (128, 65, 1280, 2)
Positional embedding: (1, 16384, 1280)
Missing keys: []
Unexpected keys: []

Model distribution:
Embedding: cuda:0
Spectral block 0: cuda:0
Spectral block 1: cuda:0
Attention block 0: cuda:0
Attention block 1: cuda:0
Attention block 2: cuda:1
Attention block 3: cuda:1
Attention block 4: cuda:1
Attention block 5: cuda:1
Attention block 6: cuda:1
Attention block 7: cuda:1
Decoder: cuda:1


In [26]:
dummy_batch = {
    "ts": torch.zeros(
        1, 1, 1, 2048, 2048,
        device="cuda:0"
    ),
    "time_delta_input": torch.zeros(
        1, 1,
        device="cuda:0"
    ),
}

with torch.no_grad():
    tokens = surya_backbone.embedding.patch_embed(
        dummy_batch["ts"]
    )

print("Patch tokens:", tokens.shape)

print(
    "Positional embedding:",
    surya_backbone.embedding.pos_embed.shape
)

for i, block in enumerate(
    surya_backbone.backbone.blocks_spectral_gating
):
    print(
        f"Spectral filter {i}:",
        block.filter.complex_weight.shape
    )

Patch tokens: torch.Size([1, 16384, 1280])
Positional embedding: torch.Size([1, 16384, 1280])
Spectral filter 0: torch.Size([128, 65, 1280, 2])
Spectral filter 1: torch.Size([128, 65, 1280, 2])


In [27]:
surya_backbone.eval()

with torch.no_grad():
    out = surya_backbone(dummy_batch)

print("Output type:", type(out))

if torch.is_tensor(out):
    print("Output shape:", out.shape)
elif isinstance(out, (list, tuple)):
    for i, x in enumerate(out):
        print(i, type(x), getattr(x, "shape", None))

Output type: <class 'torch.Tensor'>
Output shape: torch.Size([1, 1, 2048, 2048])


# Experiments
## Experiment Utilities

In [28]:
def run_experiment(
    name,
    model,
    epochs=20,
    learning_rate=1e-4,
    patience=3
):
    checkpoint_path = (
        CHECKPOINT_DIR / f"{name}.pt"
    )

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

    parameter_count = sum(
        parameter.numel()
        for parameter in model.parameters()
    )

    model = prepare_model(model)

    print(f"Experiment: {name}")
    print(f"Parameters: {parameter_count:,}")
    print(f"Checkpoint: {checkpoint_path}")

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=learning_rate
    )

    history = train_model(
        model=model,
        train_loader=train_loader,
        validation_loader=validation_loader,
        optimizer=optimizer,
        criterion=segmentation_loss,
        device=device,
        epochs=epochs,
        checkpoint_path=checkpoint_path,
        patience=patience
    )

    return {
        "name": name,
        "history": history,
        "checkpoint_path": checkpoint_path,
        "parameters": parameter_count
    }

## Run Finetune

In [ ]:
surya_vanilla = run_experiment(
    name="surya_vanilla",
    model=surya_backbone,
    epochs=20,
    learning_rate=1e-4,
    patience=3,
)

Experiment: surya_vanilla
Parameters: 227,578,368
Checkpoint: /kaggle/working/checkpoints/surya_vanilla.pt

Epoch 1/20
Training image 1/905
Training image 2/905
Training image 3/905
Training image 4/905
Training image 5/905
Training image 6/905
Training image 7/905
Training image 8/905
Training image 9/905
Training image 10/905
Training image 11/905
Training image 12/905
Training image 13/905
Training image 14/905
Training image 15/905
Training image 16/905
Training image 17/905
Training image 18/905
Training image 19/905
Training image 20/905
Training image 21/905
Training image 22/905
Training image 23/905
Training image 24/905
Training image 25/905
Training image 26/905
Training image 27/905
Training image 28/905
Training image 29/905
Training image 30/905
Training image 31/905
Training image 32/905
Training image 33/905
Training image 34/905
Training image 35/905
Training image 36/905
Training image 37/905
Training image 38/905
Training image 39/905
Training image 40/905
Training i

Validation:   0%|          | 0/249 [00:00<?, ?it/s]

Train Loss:      0.9345
Validation Loss: 0.7452
Dice:            0.2746
IoU:             0.1684

PQ:              0.0237
├─ SQ:           0.5643
└─ RQ:           0.0420

Instances
├─ TP:           92
├─ FP:           2442
├─ FN:           1760
├─ One-to-many:  280
└─ Many-to-one:  5
Saved new best model (PQ: 0.0237).

Epoch 2/20
Training image 1/905
Training image 2/905
Training image 3/905
Training image 4/905
Training image 5/905
Training image 6/905
Training image 7/905
Training image 8/905
Training image 9/905
Training image 10/905
Training image 11/905
Training image 12/905
Training image 13/905
Training image 14/905
Training image 15/905
Training image 16/905
Training image 17/905
Training image 18/905
Training image 19/905
Training image 20/905
Training image 21/905
Training image 22/905
Training image 23/905
Training image 24/905
Training image 25/905
Training image 26/905
Training image 27/905
Training image 28/905
Training image 29/905
Training image 30/905
Training image 31/

Validation:   0%|          | 0/249 [00:00<?, ?it/s]

Train Loss:      0.6591
Validation Loss: 0.6591
Dice:            0.3619
IoU:             0.2314

PQ:              0.0320
├─ SQ:           0.5782
└─ RQ:           0.0554

Instances
├─ TP:           202
├─ FP:           5238
├─ FN:           1650
├─ One-to-many:  499
└─ Many-to-one:  12
Saved new best model (PQ: 0.0320).

Epoch 3/20
Training image 1/905
Training image 2/905
Training image 3/905
Training image 4/905
Training image 5/905
Training image 6/905
Training image 7/905
Training image 8/905
Training image 9/905
Training image 10/905
Training image 11/905
Training image 12/905
Training image 13/905
Training image 14/905
Training image 15/905
Training image 16/905
Training image 17/905
Training image 18/905
Training image 19/905
Training image 20/905
Training image 21/905
Training image 22/905
Training image 23/905
Training image 24/905
Training image 25/905
Training image 26/905
Training image 27/905
Training image 28/905
Training image 29/905
Training image 30/905
Training image 3

Validation:   0%|          | 0/249 [00:00<?, ?it/s]

... (20 outputs left out)

Train Loss:      0.4024
Validation Loss: 0.5699
Dice:            0.4460
IoU:             0.3023

PQ:              0.1084
├─ SQ:           0.6049
└─ RQ:           0.1792

Instances
├─ TP:           427
├─ FP:           2486
├─ FN:           1425
├─ One-to-many:  348
└─ Many-to-one:  9
Saved new best model (PQ: 0.1084).

Epoch 14/20
Training image 1/905
Training image 2/905
Training image 3/905
Training image 4/905
Training image 5/905
Training image 6/905
Training image 7/905
Training image 8/905
Training image 9/905
Training image 10/905
Training image 11/905
Training image 12/905
Training image 13/905
Training image 14/905
Training image 15/905
Training image 16/905
Training image 17/905
Training image 18/905
Training image 19/905
Training image 20/905
Training image 21/905
Training image 22/905
Training image 23/905
Training image 24/905
Training image 25/905
Training image 26/905
Training image 27/905
Training image 28/905
Training image 29/905
Training image 30/905
Training image 3

Validation:   0%|          | 0/249 [00:00<?, ?it/s]

Train Loss:      0.3834
Validation Loss: 0.5511
Dice:            0.4617
IoU:             0.3156

PQ:              0.1279
├─ SQ:           0.6041
└─ RQ:           0.2116

Instances
├─ TP:           447
├─ FP:           1925
├─ FN:           1405
├─ One-to-many:  289
└─ Many-to-one:  14
Saved new best model (PQ: 0.1279).

Epoch 15/20
Training image 1/905
Training image 2/905
Training image 3/905
Training image 4/905
Training image 5/905
Training image 6/905
Training image 7/905
Training image 8/905
Training image 9/905
Training image 10/905
Training image 11/905
Training image 12/905
Training image 13/905
Training image 14/905
Training image 15/905
Training image 16/905
Training image 17/905
Training image 18/905
Training image 19/905
Training image 20/905
Training image 21/905
Training image 22/905
Training image 23/905
Training image 24/905
Training image 25/905
Training image 26/905
Training image 27/905
Training image 28/905
Training image 29/905
Training image 30/905
Training image 

Validation:   0%|          | 0/249 [00:00<?, ?it/s]